[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TunaLee/posco/blob/main/notebooks/day13_live.ipynb)

# Day 13 · 강의 — MCP 서버 만들기 · 앱에 붙이기

노트북 안에 있던 서버를 파일로 떼어 내고, 띄우고, 다른 앱이 붙게 한다

---

### 시작하기 전에

1. **파일 → 드라이브에 사본 저장** 을 먼저 누른다. 안 하면 고친 내용이 남지 않는다.
2. 셀을 고르고 **Shift + Enter** 로 실행한다.

수업을 따라가며 진행한다.

모든 셀에 **코드가 채워져 있다.** 위에서부터 실행해 결과를 눈으로 확인한다.
강사가 설명하는 동안 값을 바꿔 가며 돌려 본다.

문제는 실행하면 `assert` 로 자가 채점된다. 맞으면 `통과` 가 찍히고,
틀리면 기대값과 실제값이 같이 나온다.

## 1. 서버를 파일로 떼어 내기

어제 만든 서버는 **노트북 안에** 있었다. `Client(mcp)` 로 붙었다.
노트북을 닫으면 사라지고, 나 말고는 아무도 못 쓴다.

오늘은 그것을 **파일로 떼어 내고, 진짜로 띄우고, 다른 앱이 붙게** 한다.

In [ ]:
# FastMCP 를 받는다
!pip install -q fastmcp

import json, socket, threading, time, urllib.request
import pandas as pd
from fastmcp import Client, FastMCP

df = pd.read_csv('https://tunalee.github.io/posco/data/cell_process.csv')
print('%d행 %d열' % df.shape)

### 서버를 파일로 쓴다

아래 글자를 그대로 `server.py` 에 적는다. 어제 노트북에 있던 것과 몸통이 같다.

In [ ]:
# 파일 하나가 곧 서버다
SERVER = '''
import pandas as pd
from fastmcp import FastMCP

df = pd.read_csv("https://tunalee.github.io/posco/data/cell_process.csv")
mcp = FastMCP("공정 도우미")

OPEN = ["로트번호", "시각", "설비호기", "교대조", "판정"]   # 내보내도 되는 칸
LIMIT = 20                                                # 한 번에 주는 최대 행수

@mcp.tool()
def defect_rate(machine: str, shift: str = "") -> str:
    "설비호기의 불량률을 돌려준다. 교대조를 주면 그 안에서만 센다."
    d = df[df["설비호기"] == machine]
    if shift:
        d = d[d["교대조"] == shift]
    if not len(d):
        return "해당 조건에 데이터가 없다"
    bad = int((d["판정"] == "불량").sum())
    return "%s %s 측정 %d건 중 불량 %d건 · %.1f%%" % (
        machine, shift or "전체", len(d), bad, 100.0 * bad / len(d))

@mcp.tool()
def recent_lots(machine: str, n: int = 5) -> str:
    "설비의 최근 로트 기록을 돌려준다. 공정 조건값은 내보내지 않는다."
    d = df[df["설비호기"] == machine].tail(min(n, LIMIT))
    return d[OPEN].to_string(index=False) if len(d) else "해당 설비가 없다"

if __name__ == "__main__":
    mcp.run()
'''
open('server.py', 'w', encoding='utf-8').write(SERVER)
print(open('server.py', encoding='utf-8').read()[:300])

**맨 아래 `mcp.run()` 이 어제와 갈리는 자리다.** 이 줄이 서버를 돌린다.
`__name__ == "__main__"` 은 「이 파일을 직접 실행했을 때만」이라는 뜻이다.

## 2. 띄우고 주소 얻기

`mcp.run()` 은 **셀을 붙잡고 안 놓는다.** 서버는 원래 계속 도는 것이라 그렇다.
노트북에서 이어서 쓰려면 **딴 갈래로 띄운다.**

In [ ]:
# 같은 서버를 이번엔 HTTP 로 띄운다. 딴 갈래로 돌려야 셀이 안 막힌다.
srv = FastMCP('공정 도우미')

@srv.tool()
def defect_rate(machine: str, shift: str = '') -> str:
    '''설비호기의 불량률을 돌려준다. 교대조를 주면 그 안에서만 센다.

    machine: 설비호기. 1호기 ~ 4호기
    shift: 교대조. 주간 또는 야간. 비우면 전체
    '''
    d = df[df['설비호기'] == machine]
    if shift:
        d = d[d['교대조'] == shift]
    if not len(d):
        return '해당 조건에 데이터가 없다'
    bad = int((d['판정'] == '불량').sum())
    return '%s %s 측정 %d건 중 불량 %d건 · %.1f%%' % (
        machine, shift or '전체', len(d), bad, 100.0 * bad / len(d))

PORT = 8931
threading.Thread(target=lambda: srv.run(transport='http', host='127.0.0.1', port=PORT),
                 daemon=True).start()

for _ in range(60):                       # 포트가 열릴 때까지 기다린다
    s = socket.socket(); s.settimeout(0.3)
    if s.connect_ex(('127.0.0.1', PORT)) == 0:
        break
    time.sleep(0.3)

ADDR = 'http://127.0.0.1:%d/mcp' % PORT
print('서버가 떴다 —', ADDR)

**주소가 생겼다.** 이제 이 서버는 노트북 안의 변수가 아니라 **주소가 있는 것**이다.

## 3. 주소로 붙기

어제는 서버를 담은 변수를 넘겼다. 오늘은 **주소만** 넘긴다.

In [ ]:
# 괄호 안만 바뀐다. 그 뒤는 어제와 한 글자도 안 다르다.
async def peek(target):
    async with Client(target) as c:
        names = [t.name for t in await c.list_tools()]
        out = (await c.call_tool('defect_rate', {'machine': '3호기'})).content[0].text
        return names, out

print('객체로 붙기 :', await peek(srv))
print('주소로 붙기 :', await peek(ADDR))

**둘이 똑같다.** `Client()` 괄호 안만 바뀌었다.

이게 MCP 가 하려던 것이다 &mdash; 서버가 **어디에 있든** 붙는 쪽 코드는 같다.
노트북 안이든, 옆자리 PC 든, 사내 서버든.

> **실습문제 1.** 포트를 **8932** 로 바꿔 서버를 하나 더 띄우고, 주소로 붙어 본다.
> 같은 기계에 서버 둘이 동시에 돈다. 포트가 이름표다.

In [ ]:
srv2 = FastMCP('둘째 서버')

@srv2.tool()
def machine_list() -> str:
    '''쓸 수 있는 설비호기 이름을 모두 돌려준다'''
    return ', '.join(sorted(df['설비호기'].unique()))

PORT2 = 8932
threading.Thread(target=lambda: srv2.run(transport='http', host='127.0.0.1', port=PORT2),
                 daemon=True).start()

for _ in range(60):
    s = socket.socket(); s.settimeout(0.3)
    if s.connect_ex(('127.0.0.1', PORT2)) == 0: break
    time.sleep(0.3)
async with Client('http://127.0.0.1:%d/mcp' % PORT2) as c:
    print('붙은 도구:', [t.name for t in await c.list_tools()])

## 4. 모델에게 넘어가는 것

도구를 만들면 모델에게 무엇이 넘어가는지부터 본다. **여기 없는 것은 모델이 모른다.**

In [ ]:
# 어제 쓰던 키를 그대로 쓴다
import getpass
KEY = getpass.getpass('nvapi- 로 시작하는 키: ')

URL = 'https://integrate.api.nvidia.com/v1/chat/completions'
MODEL = 'nvidia/llama-3.3-nemotron-super-49b-v1'

def chat(messages, tools=None, n=600):
    body = {'model': MODEL, 'max_tokens': n, 'temperature': 0, 'messages': messages}
    if tools:
        body['tools'] = tools
    req = urllib.request.Request(URL, data=json.dumps(body).encode(), headers={
        'Authorization': 'Bearer ' + KEY,
        'Content-Type': 'application/json', 'Accept': 'application/json'})
    with urllib.request.urlopen(req, timeout=180) as f:
        return json.load(f)['choices'][0]['message']

In [ ]:
# 서버가 모델에게 넘기는 글을 그대로 찍어 본다
async with Client(srv) as c:
    t = (await c.list_tools())[0]
print('이름   ', t.name)
print('설명   ', t.description)
print('스키마 ', json.dumps(t.inputSchema, ensure_ascii=False, indent=2))

**이름 · 설명 · 스키마 셋이 전부다.** 함수 몸통은 안 넘어간다.

그래서 코드가 아무리 정확해도 **이름과 설명이 흐리면** 모델은 못 고른다.
설명은 사람이 아니라 **모델이 읽는 글**이다.

In [ ]:
# 어제 만든 고리 그대로. 클라이언트가 목록을 받고, 모델이 고른다.
def to_openai(tools):
    return [{'type': 'function',
             'function': {'name': t.name, 'description': t.description,
                          'parameters': t.inputSchema}}
            for t in tools]

async def run(target, question, log=True):
    called = []
    async with Client(target) as client:
        spec = to_openai(await client.list_tools())
        messages = [{'role': 'system', 'content':
                     '너는 공정 데이터를 보는 비서다. 한국어로만 답한다. '
                     '숫자는 도구로 조회한 값만 쓴다.'},
                    {'role': 'user', 'content': question}]
        for _ in range(4):
            m = chat(messages, spec)
            messages.append(m)
            calls = m.get('tool_calls') or []
            if not calls:
                return (m.get('content') or '').strip() or '[도구를 안 불렀다]', called
            for c in calls:
                name = c['function']['name']
                args = json.loads(c['function']['arguments'] or '{}')
                called.append(name)
                if log:
                    print('  [MCP] %s(%s)' % (name, args))
                try:
                    out = (await client.call_tool(name, args)).content[0].text
                except Exception as e:          # 터져도 그 말을 모델에게 넘긴다
                    out = '도구 실행 실패: %s' % e
                messages.append({'role': 'tool', 'tool_call_id': c['id'], 'content': out})
    return '[한도]', called

## 5. 도구 고르기가 흔들리는 것

도구가 둘 이상이면 모델이 **고른다.** 그런데 이 고르기는 매번 같지 않다.
같은 질문을 세 번 던져 본다.

In [ ]:
# 이름이 비슷한 도구 둘을 한 서버에 둔다
pick = FastMCP('고르기 시험')

@pick.tool()
def check_quality(machine: str) -> str:
    '''설비의 불량률을 돌려준다. 품질·불량·양품을 물으면 이것을 쓴다.

    machine: 설비호기. 1호기 ~ 4호기
    '''
    d = df[df['설비호기'] == machine]
    bad = int((d['판정'] == '불량').sum())
    return '%s 측정 %d건 중 불량 %d건 · %.1f%%' % (machine, len(d), bad,
                                              100.0 * bad / len(d))

@pick.tool()
def check_runtime(machine: str) -> str:
    '''설비가 얼마나 돌았는지 가동시간을 돌려준다. 가동·시간을 물으면 이것을 쓴다.

    machine: 설비호기. 1호기 ~ 4호기
    '''
    return '%s 가동 %d분' % (machine, len(df[df['설비호기'] == machine]) * 3)

print('도구 둘을 붙였다')

In [ ]:
# 뜻이 분명한 질문과 모호한 질문을 각각 세 번씩
for q in ['3호기 불량률 알려줘', '3호기 상태 좀 봐줘']:
    print('Q', q)
    for i in range(3):
        _, called = await run(pick, q, log=False)
        print('   %d회 → %s' % (i + 1, called))
    print()

**「불량률」처럼 뜻이 분명하면 세 번 다 같은 도구가 불린다.**
**「상태 좀 봐줘」처럼 모호하면 매번 다르다.** 온도를 0 으로 놓아도 그렇다.

그래서 도구는 **좁게, 이름을 분명하게** 만든다.
한 도구가 여러 일을 하면 모델이 언제 부를지 판단할 근거가 없다.

## 6. 나쁜 입력 막기

모델은 **없는 설비 이름**을 넣기도 한다. 그때 도구가 무엇을 돌려주느냐로 갈린다.
먼저 도구를 직접 불러 **모델이 받게 될 글**을 본다.

In [ ]:
# 검증이 없는 도구와, 길을 알려 주는 도구
rude = FastMCP('막 만든 서버')
@rude.tool()
def defect_rate(machine: str) -> str:
    '''설비호기의 불량률을 돌려준다'''
    d = df[df['설비호기'] == machine]
    bad = int((d['판정'] == '불량').sum())
    return '%.1f%%' % (100.0 * bad / len(d))        # 없는 설비면 0 으로 나눈다

kind = FastMCP('친절한 서버')
@kind.tool()
def defect_rate(machine: str) -> str:
    '''설비호기의 불량률을 돌려준다. 설비 품질을 물으면 이것을 쓴다.

    machine: 설비호기. 1호기 ~ 4호기
    '''
    ok = sorted(df['설비호기'].unique())
    if machine not in ok:
        return '「%s」 는 없는 설비다. 쓸 수 있는 이름: %s' % (machine, ', '.join(ok))
    d = df[df['설비호기'] == machine]
    bad = int((d['판정'] == '불량').sum())
    return '%s 측정 %d건 중 불량 %d건 · %.1f%%' % (machine, len(d), bad,
                                              100.0 * bad / len(d))

print('준비됐다')

In [ ]:
# 없는 설비를 넣어 도구를 직접 불러 본다
async def raw(server, machine):
    async with Client(server) as c:
        try:
            return (await c.call_tool('defect_rate', {'machine': machine})).content[0].text
        except Exception as e:
            return '터졌다 → %s' % e

print('검증 없음   :', await raw(rude, '7호기'))
print('길을 알려 줌 :', await raw(kind, '7호기'))

> 검증이 없는 쪽은 **빨간 오류 상자**가 길게 찍힌다. 서버가 남기는 기록이라 그렇다.
> 놀랄 것 없다. 볼 것은 그 아래 한 줄이다.

**이 두 줄이 모델에게 넘어가는 글이다.**

앞엣것은 무엇이 잘못됐는지 안 알려 준다. 뒤엣것은 **쓸 수 있는 이름을 같이 준다.**
에러 문구는 사람만 읽는 것이 아니다.

In [ ]:
# 모델에게 맡기면 어떻게 되나
for name, server in [('검증 없음', rude), ('길을 알려 줌', kind)]:
    answer, called = await run(server, '7호기 불량률 알려줘', log=False)
    print('—', name, '· 부른 도구', called)
    print('  ', answer[:200].replace(chr(10), ' '))
    print()

> **답에 「가상 데이터」나 「예시」 같은 말이 섞이면 위험 신호다.**
> 도구가 값을 안 주면 모델은 **지어내서라도** 답하려 든다.
> 없으면 없다고 돌려주는 것이 도구의 몫이다.

## 7. 열 것과 안 열 것

여기까지는 **읽기만** 하는 도구였다. 쓰는 도구를 붙이는 순간 성격이 달라진다.
읽기는 틀려도 되돌릴 수 있지만, 쓰기는 안 된다.

In [ ]:
# 읽기 도구와 쓰기 도구를 표시로 갈라 둔다
memo = FastMCP('점검 메모')
NOTES = []

@memo.tool(annotations={'readOnlyHint': True})
def list_notes() -> str:
    '''적어 둔 점검 메모를 모두 돌려준다'''
    return '\n'.join('%d. %s' % (i + 1, n) for i, n in enumerate(NOTES)) or '메모가 없다'

@memo.tool(annotations={'readOnlyHint': False, 'destructiveHint': False})
def add_note(text: str) -> str:
    '''점검 메모를 새로 적는다. 기존 메모는 건드리지 않는다.

    text: 적을 내용
    '''
    NOTES.append(text)
    return '적었다. 지금 메모 %d개' % len(NOTES)

async with Client(memo) as c:
    for t in await c.list_tools():
        ann = t.annotations
        print('%-12s 읽기전용 %s' % (t.name, getattr(ann, 'readOnlyHint', None) if ann else None))

**`readOnlyHint` 는 붙는 앱에게 주는 표시다.** 강제가 아니다.
앱은 이걸 보고 「이건 그냥 실행, 이건 사람에게 물어보고」를 가른다.

> 표시를 믿고 위험한 것을 여는 것이 아니다. **막는 것은 도구 안에서** 해야 한다.

In [ ]:
# 메모를 남기게 하고, 도구가 쥔 진짜 내용과 견준다
print(await run(memo, '3호기 베어링 소음 있다고 메모 남겨줘', log=True)[0]
      if False else (await run(memo, '3호기 베어링 소음 있다고 메모 남겨줘', log=True))[0])
print()
print('도구가 쥔 진짜 메모 —', NOTES)
answer, _ = await run(memo, '메모 목록 보여줘', log=False)
print('모델이 한 말 —', answer[:200].replace(chr(10), ' '))

**도구가 쥔 것은 한 줄인데 모델이 여러 줄을 말하면 지어낸 것이다.**
담당자 이름이나 날짜처럼 **그럴듯한 것**이 붙어 나오면 특히 그렇다.

권한을 정하는 것은 시스템 프롬프트가 아니다. **도구 목록 그 자체다** &mdash;
지우는 도구를 안 만들면 모델은 못 지운다. 아무리 잘 설득해도 못 지운다.

## 8. 서버 여럿 붙이기

조마다 서버를 만들면 **한 앱에 여럿을 붙여야** 한다. 여기서 사고가 난다.

In [ ]:
# 두 조가 각각 서버를 냈다. 그런데 도구 이름이 겹친다.
team1 = FastMCP('1조 설비')
@team1.tool()
def defect_rate(machine: str) -> str:
    '''설비호기의 불량률을 돌려준다'''
    return '[1조] %s 12.0%%' % machine

team2 = FastMCP('2조 품질')
@team2.tool()
def find_rule(question: str) -> str:
    '''규정을 찾는다'''
    return '[2조] 제26조(작업중지 등)'
@team2.tool()
def defect_rate(machine: str) -> str:       # 1조와 이름이 같다
    '''설비호기의 불량률을 돌려준다'''
    return '[2조] %s 99.9%%' % machine

print('두 조 다 defect_rate 를 만들었다')

In [ ]:
# 접두어 없이 그냥 붙여 본다
plain = FastMCP('접두어 없는 호스트')
plain.mount(team1)
plain.mount(team2)

async with Client(plain) as c:
    print('도구 목록:', [t.name for t in await c.list_tools()])
    print('불러 보면:', (await c.call_tool('defect_rate', {'machine': '3호기'})).content[0].text)

**1조 도구가 사라졌다.** 경고 한 줄이 로그에 찍힐 뿐 목록에서는 그냥 없다.
부르면 2조 것이 나온다. 1조는 자기 도구가 안 불린다는 것도 모른다.

In [ ]:
# 접두어를 붙이면 둘 다 산다
merged = FastMCP('사내 호스트')
merged.mount(team1, namespace='team1')
merged.mount(team2, namespace='team2')

async with Client(merged) as c:
    print('도구 목록:', [t.name for t in await c.list_tools()])
    for n in ['team1_defect_rate', 'team2_defect_rate']:
        print(n, '→', (await c.call_tool(n, {'machine': '3호기'})).content[0].text)

**도구 이름은 앱 안에서 전역이다.** 서버가 달라도 이름이 같으면 부딪힌다.
붙이는 쪽이 접두어를 붙여 갈라 준다.

## 9. 컨테이너로 싸기

서버가 **한 대에서만** 도는 문제가 남았다.
파이썬 버전, 깔린 꾸러미, 데이터 경로가 사람마다 다르다.

컨테이너는 **그 전부를 한 덩이로 싸는** 것이다.

In [ ]:
# 서버가 쓰는 꾸러미를 적어 둔다
open('requirements.txt', 'w').write('fastmcp==3.4.7\npandas\n')

DOCKERFILE = '''
FROM python:3.12-slim

WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY server.py .
EXPOSE 8000
CMD ["fastmcp", "run", "server.py",
     "--transport", "http", "--host", "0.0.0.0", "--port", "8000"]
'''
open('Dockerfile', 'w').write(DOCKERFILE.strip() + '\n')
print(open('Dockerfile').read())

한 줄씩 무엇인지 &mdash;

**FROM** 바탕이 될 이미지. 파이썬 3.12 가 깔린 최소 리눅스다.

**COPY** 파일을 이미지 안으로 넣는다.

**RUN** 이미지를 만들 때 한 번 도는 명령. 꾸러미를 여기서 깐다.

**CMD** 컨테이너가 뜰 때 도는 명령.

> `server.py` 는 `mcp.run()` 한 줄로 끝난다. **어떻게 띄울지는 상자가 정한다** &mdash;
> 그래서 같은 파일이 노트북에서도, 컨테이너에서도 그대로 돈다.

## 10. compose 와 사내망

조가 셋이면 서버가 셋이다. 하나씩 띄우고 포트를 외우는 것은 오래 못 간다.
**compose 는 여럿을 한 파일에 적고 한 번에 올린다.**

In [ ]:
# 호스트 하나에 조별 서버 둘을 붙이는 구성
COMPOSE = '''
services:
  team1:
    build: ./teams/team1
    expose: ["8000"]

  team2:
    build: ./teams/team2
    expose: ["8000"]

  host:
    build: ./host
    ports: ["8080:8080"]
    environment:
      NVIDIA_API_KEY: ${NVIDIA_API_KEY}
      TEAM_URLS: "http://team1:8000/mcp,http://team2:8000/mcp"
    depends_on: [team1, team2]
'''
open('compose.yml', 'w').write(COMPOSE.strip() + '\n')
print(open('compose.yml').read())

**`http://team1:8000/mcp`** &mdash; 여기서 `team1` 은 기계 이름이 아니라 **서비스 이름**이다.
compose 가 서로를 이름으로 찾게 이어 준다. IP 를 적을 일이 없다.

밖으로 연 것은 `host` 의 8080 하나뿐이다. **조별 서버는 밖에서 못 붙는다** &mdash;
`expose` 는 안쪽끼리만 열고, `ports` 는 바깥까지 연다.

### 사내망에서 쓰려면

바깥 인터넷이 막힌 곳에서는 위 그대로는 안 된다. `FROM python:3.12-slim` 부터 못 받는다.
막히는 자리가 셋이고, 자리마다 푸는 법이 있다.

In [ ]:
# ① 이미지를 사내 저장소에서 받는다  ② 꾸러미도 사내 미러에서 받는다
INTRA = '''
FROM harbor.사내주소/library/python:3.12-slim

WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir \\
      --index-url https://nexus.사내주소/repository/pypi/simple \\
      --trusted-host nexus.사내주소 \\
      -r requirements.txt

COPY server.py .
CMD ["python", "server.py"]
'''
open('Dockerfile.intra', 'w').write(INTRA.strip() + '\n')
print(open('Dockerfile.intra').read())

**③ 저장소도 미러도 없으면** 바깥에서 만들어 파일로 옮긴다.

```
바깥 PC   docker build -t 공정도우미:1.0 .
          docker save 공정도우미:1.0 -o 공정도우미.tar
                     ↓  USB · 승인된 반입 경로
사내 PC   docker load -i 공정도우미.tar
          docker run -p 8000:8000 공정도우미:1.0
```

이 방법은 **빌드를 사내에서 안 한다.** 다 만든 것을 통째로 옮긴다.
폐쇄망에서 제일 흔하게 쓰는 길이다.

> 프록시만 있는 곳이라면 빌드할 때 넘겨준다 &mdash;
> `docker build --build-arg HTTP_PROXY=$HTTP_PROXY --build-arg HTTPS_PROXY=$HTTPS_PROXY .`

## 11. 조별 서버 설계

발표에 낼 것을 여기서 정한다. **호스트는 준비돼 있다.**
조는 `server.py` 하나만 내면 채팅창에 붙는다.

### 프롬프트로 옮기면

위 다섯 칸을 Codex 에 이렇게 넘긴다.

```
FastMCP 로 MCP 서버 파일 하나를 만들어 줘. 파일명은 server.py.

서버 이름: 점검 도우미
데이터: cell_process.csv 를 pandas 로 읽는다

도구 셋:
  1. 최근 점검 이력 조회 — 설비호기와 건수를 받는다. 최대 20건.
  2. 작업표준 검색 — 찾을 내용을 받아 관련 조항 셋을 돌려준다.
  3. 점검 메모 남기기 — 적을 내용을 받는다. 지우는 기능은 만들지 마라.

지켜야 할 것:
  - 설비 원시 계측값과 담당자 이름은 어떤 도구도 내보내지 않는다
  - 없는 설비를 물으면 쓸 수 있는 이름을 알려 준다
  - 각 도구 설명에 「언제 쓰는지」를 한 줄 적는다
  - 맨 아래에 mcp.run() 을 둔다
```

**「지켜야 할 것」이 오늘 배운 전부다.** 경계 · 친절한 에러 · 설명.

### 오늘 손에 남는 것

**하나** &mdash; `mcp.run()` 한 줄이 서버를 띄우고, 그 순간 **주소가 생긴다**.

**둘** &mdash; `Client()` 괄호 안만 바뀐다. 서버가 어디에 있든 붙는 쪽은 같다.

**셋** &mdash; 설명은 사람이 아니라 **모델이 읽는 글**이다. 「언제 쓰는지」를 적는다.

**넷** &mdash; 에러 문구도 모델이 읽는다. 길을 알려 주면 스스로 고쳐 부른다.

**다섯** &mdash; 권한은 프롬프트가 아니라 **만들 도구를 고르는 것**으로 정한다.

**여섯** &mdash; 도구 이름은 전역이다. 서버를 여럿 붙일 때는 접두어로 가른다.

**일곱** &mdash; 컨테이너는 파이썬·꾸러미·데이터를 한 덩이로 싼다. compose 는 그 덩이 여럿을 잇는다.